In [1]:
import h5py
import torch
from torch import nn
from torchsummary import summary
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from chess2.bot import ChessDataset
from chess2.bot import NeuralNetwork

In [2]:
# Hyperparameters
BATCH_SIZE = 64
EPOCHS = 25 #150
LEARNING_RATE = 1e-3 #1e-4 was good
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 5

In [3]:
train_data = ChessDataset("/Users/jonas/coding/python/chess2/src/chess2/bot/data_leela/chess_data_list.pkl", 0, 200000)
validation_data = ChessDataset("/Users/jonas/coding/python/chess2/src/chess2/bot/data_leela/chess_data_list.pkl", 300000, 350000)

In [4]:
train_dataloader = DataLoader(train_data, BATCH_SIZE, shuffle=True)
validation_dataloader = DataLoader(validation_data, BATCH_SIZE, shuffle=True)

In [5]:
device = torch.device("cpu")
model = NeuralNetwork().to(device)

In [6]:
loss_policy = nn.CrossEntropyLoss()

In [7]:
optimizer = torch.optim.SGD(
    params=model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    momentum=0.9,
    nesterov=True
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

# scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     optimizer=optimizer,
#     T_0=10,         # Restart after 10 epochs
#     T_mult=2,       # Restart time every time
#     eta_min=1e-6    # minimum learning rate
# )

In [8]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)

    # Set the model to training mode - important for batch normalization and dropout layers
    model.train()

    for batch, (boards, flags, prob_idx) in enumerate(dataloader):
        if any(i > 1858 for i in prob_idx):
            raise ValueError("Invalid Policy Index")
        move_pred = model(boards, flags)
        # move_pred = move_pred.masked_fill(~moves_mask.bool(), -1e9)

        optimizer.zero_grad()

        loss = loss_fn(move_pred, prob_idx)

        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * BATCH_SIZE
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [9]:
def validation_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    validation_loss, correct = 0, 0

    # Set the model in evaluation mode
    model.eval()

    # no_grad to suppress gradient computation
    with torch.no_grad():
        for (boards, flags, prob_idx) in dataloader:
            move_pred = model(boards, flags)

            validation_loss += loss_fn(move_pred, prob_idx).item()
            correct += (move_pred.argmax(dim=1).long() == prob_idx).type(torch.float).sum().item()

    validation_loss /= num_batches
    correct /= size

    print(f"Validation Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss per batch: {validation_loss:>8f} \n")

    return validation_loss

In [10]:
# model.load_state_dict(torch.load("/Users/jonas/coding/python/chess2/src/chess2/bot/saved_models/model_new.pth", weights_only=True))

In [11]:
for t in range(EPOCHS):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_policy, optimizer) # train_dataloader instead of test_dataloader
    val_loss = validation_loop(validation_dataloader, model, loss_policy)
    scheduler.step(val_loss)
    print("LR:", optimizer.param_groups[0]['lr'])
print("Done!")

Epoch 1
-------------------------------
loss: 7.534793  [    0/200000]
loss: 7.439632  [ 6400/200000]
loss: 7.261796  [12800/200000]
loss: 6.970007  [19200/200000]
loss: 7.082368  [25600/200000]
loss: 7.082676  [32000/200000]
loss: 6.487836  [38400/200000]
loss: 6.714370  [44800/200000]
loss: 6.379214  [51200/200000]
loss: 6.545125  [57600/200000]
loss: 6.830485  [64000/200000]
loss: 6.323401  [70400/200000]
loss: 6.678193  [76800/200000]
loss: 6.132429  [83200/200000]
loss: 6.699227  [89600/200000]
loss: 6.097782  [96000/200000]
loss: 6.229717  [102400/200000]
loss: 6.475712  [108800/200000]
loss: 6.287694  [115200/200000]
loss: 6.545628  [121600/200000]
loss: 6.333464  [128000/200000]
loss: 6.377625  [134400/200000]
loss: 6.297211  [140800/200000]
loss: 6.361148  [147200/200000]
loss: 6.463146  [153600/200000]
loss: 6.268751  [160000/200000]
loss: 6.592525  [166400/200000]
loss: 6.633180  [172800/200000]
loss: 6.153519  [179200/200000]
loss: 6.121615  [185600/200000]
loss: 6.188057  

In [12]:
torch.save(model.state_dict(), '/Users/jonas/coding/python/chess2/src/chess2/bot/saved_models/model_less_data_leela.pth')